# 1 · Understand the traffic data

Run this notebook first. We count vehicles passing one road measurement point in an hour. The aim is to flag unusual measurements, then investigate how dependable the flags are.

**Before starting:** follow [the setup guide](../docs/getting-started.md), select **Python 3 (ipykernel)** and run `uv run traffic-study prepare`. In JupyterLab click a cell and press **Shift+Return**. Run from top to bottom.

In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
from traffic_anomaly.paths import ROOT, RAW, RESULTS, FIGURES, config
from traffic_anomaly.data import load_split
cfg = config()
print("Project configuration found:", (ROOT / "configs/study.json").is_file())

## Inspect the original resource

The data is from UCI. Weather descriptions can create more than one source row for the same hour. We must inspect this before treating rows as independent hourly measurements. The string `None` in `holiday` means no listed holiday; it is not a missing cell.

In [ ]:
raw = pd.read_csv(RAW / "Metro_Interstate_Traffic_Volume.csv.gz", keep_default_na=False)
print("Raw shape:", raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("type"))

## Read the audit

The audit collapses identical timestamp/count pairs. A timestamp with conflicting counts would be excluded rather than guessed. The original download is preserved. Missing hours are listed separately; they do not become zero-traffic hours.

In [ ]:
audit = json.loads((RESULTS / "data_audit.json").read_text())
display(pd.Series({k:audit[k] for k in ["raw_rows","same_count_duplicates_collapsed","conflicting_timestamps_excluded","hourly_rows","missing_hours_in_full_span","longest_gap_missing_hours","zero_volume_hours_retained"]}, name="count").to_frame())

## Understand the chronological split

Training teaches the model; validation sets the alert threshold; test data provides the later assessment. We use 60%, 20% and 20% of the ordered observed hours. Because some hours are absent, these percentages are not equal fractions of elapsed calendar time.

In [ ]:
train, validation, test = [load_split(s) for s in ["train","validation","test"]]
display(pd.DataFrame(audit["split_summary"]).T)
assert train.date_time.max() < validation.date_time.min()
assert validation.date_time.max() < test.date_time.min()
print("Chronological order confirmed.")

## Explore patterns using training data

The chart shows the median count at each hour. A high count at a commuting hour is not automatically anomalous. The shaded range describes the middle half of training observations.

In [ ]:
import matplotlib.pyplot as plt
training_view = train.assign(hour=train.date_time.dt.hour, day_type=np.where(train.date_time.dt.dayofweek < 5, "Weekday", "Weekend"))
fig, ax = plt.subplots(figsize=(9,3.5))
for name, group in training_view.groupby("day_type"):
    hourly = group.groupby("hour").traffic_volume
    ax.plot(hourly.median(), label=name)
    ax.fill_between(hourly.median().index, hourly.quantile(.25), hourly.quantile(.75), alpha=.15)
ax.set(xlabel="Hour", ylabel="Vehicles per hour", title="Training data only")
ax.legend()
plt.show()

## Check your understanding

1. Why would summing repeated traffic counts be wrong?
2. Why is an absent hour different from a recorded zero?
3. Why must the baseline learn its typical counts from training data only?

Next: `02_understand_the_detectors.ipynb`.